In [ ]:
import math
from statistics import mean, stdev
from scipy.stats import t, bootstrap
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
def Fasta2Dict(filepath):
    seqs = {}
    with open(filepath, "r") as f:
        lines = [line.strip() for line in f if line.strip()]
    for i in range(len(lines)):
        if lines[i].startswith(">"):
            key = lines[i][1:]
            seqs[key] = lines[i+ 1]
    headers=dict(zip(seqs.values(),seqs.keys()))
    return seqs,headers

In [ ]:
Reads,headers = Fasta2Dict("READS.fasta")
reads = list(Reads.values())
Query_dict,Query_header = Fasta2Dict("QUERY.fasta")
Query = list(Query_dict.values())[0]
print(Query)

In [ ]:
def prefix(s,len=5,n=0):
    '''
    Return the beginning of len elements of a string
    '''
    return s[n:len+n]
def suffix(s,len=5,n=0):
    '''
    Return the end of len elements of a string
    '''
    start=-(len+n)
    if n==0:
        return s[start:]
    else:
        return s[start:-n]
    
def RvsComp(s):
    '''
    Return the reverse complement of the string
    '''
    pairs={"A":"T","T":"A","G":"C","C":"G"}
    rvs = s[::-1]
    newseq=""
    for n in rvs:
        newseq=newseq+pairs[n]
    return newseq

def getKmers(reads,len_kmer):
    '''Given reads generate kmers for the ends'''
    kmers={}
    for r in reads:
        fwdKmers=[prefix(r,len_kmer),suffix(r,len_kmer)]
        #check if kmer is in dictionary if not add it
        for k in fwdKmers:
            if kmers.get(k,None)==None:
                kmers[k]=[r]
            else:
                kmers[k].append(r)
        #Get Rvs Comp kmers
        rc = RvsComp(r)
        bkwdkmers=[prefix(rc,len_kmer),suffix(rc,len_kmer)]
        for k in bkwdkmers:
            if kmers.get(k,None)==None:
                kmers[k]=[rc]
            else:
                kmers[k].append(rc)
    return kmers

#Future Experiment test multiple windows of kmers
def get_NKmers_OneTable(reads,len_kmer,n):
    '''Get kmers for n steps. For example n of 5 would get a 5 window kmer Create a lookup table with all window kmers'''
    kmers={}
    for win in range(n+1):
        for r in reads:
            fwdKmers=[prefix(r,len_kmer,win),suffix(r,len_kmer,win)]
            #check if kmer is in dictionary if not add it
            for k in fwdKmers:
                if kmers.get(k,None)==None:
                    kmers[k]=[r]
                else:
                    kmers[k].append(r)
            #Get Rvs Comp kmers
            rc = RvsComp(r)
            bkwdkmers=[prefix(rc,len_kmer,win),suffix(rc,len_kmer,win)]
            for k in bkwdkmers:
                if kmers.get(k,None)==None:
                    kmers[k]=[rc]
                else:
                    kmers[k].append(rc)
    return kmers
def get_NKmers(reads,len_kmer,n):
    '''Get kmers for n steps. For example n of 5 would get a 5 window kmer'''
    kmers={}
    win=n
    for r in reads:
        fwdKmers=[prefix(r,len_kmer,win),suffix(r,len_kmer,win)]
        #check if kmer is in dictionary if not add it
        for k in fwdKmers:
            if kmers.get(k,None)==None:
                kmers[k]=[r]
            else:
                kmers[k].append(r)
        #Get Rvs Comp kmers
        rc = RvsComp(r)
        bkwdkmers=[prefix(rc,len_kmer,win),suffix(rc,len_kmer,win)]
        for k in bkwdkmers:
            if kmers.get(k,None)==None:
                kmers[k]=[rc]
            else:
                kmers[k].append(rc)
    return kmers
    

In [ ]:
def NumReadsStats(kmers_dict):
    '''Get the mean number of reads per kmer'''
    reads=list(kmers_dict.values())
    total_reads = np.array([len(r) for r in reads])
    mean=total_reads.mean()
    std=total_reads.std(ddof=1)
    print("mean:",mean)
    boot = bootstrap((total_reads,),statistic=np.mean,n_resamples=1000,confidence_level=0.95,method="percentile",random_state=0)
    print("95% CI:",boot.confidence_interval.low, boot.confidence_interval.high)
    return {"Mean_reads_kmer": mean,"ci_lower":boot.confidence_interval.low,"ci_upper": boot.confidence_interval.high}


In [ ]:
test = reads[0]
print("Orig:", test)
print("pref ",prefix(test,6))
print("suff ",suffix(test,6))
print("pref n1",prefix(test,6,1))
print("suff n1",suffix(test,6,1))
print("pref n2",prefix(test,6,2))
print("suff n2",suffix(test,6,2))
len(test)
print("Orig:   ",test)
print("RvsOrig:",test[::-1])
print("RvsComp:",RvsComp(test))

In [ ]:
#get unique kmers
len_kmer=[3,4,5,6,7,8,9,10]
results=[]
for k in tqdm(len_kmer):
    kmers=getKmers(reads,k)
    stats=NumReadsStats(kmers)
    stats["k"] = k
    results.append(stats)

kmer_exp= pd.DataFrame(results)
print(kmer_exp)
kmer_exp.to_csv("kmer_exp.csv")

In [ ]:
kmer_exp = pd.read_csv("kmer_exp.csv")

In [ ]:
plt.errorbar(kmer_exp["k"],kmer_exp["Mean_reads_kmer"],yerr=[kmer_exp["Mean_reads_kmer"]-kmer_exp["ci_lower"],kmer_exp["ci_upper"]-kmer_exp["Mean_reads_kmer"]],fmt='o')
plt.xlabel("Kmer Length (nucleotides)")
plt.ylabel("Mean Reads Per Kmer")
plt.title("Terminal Mean Reads Per Kmer Vs. Kmer Length")
plt.show()

In [ ]:
#get unique kmers
len_kmer=[5,6,7,8]
windows=[0,1,2,3,4,5]
results=[]
for k in tqdm(len_kmer):
    for w in windows:
        kmers=get_NKmers_OneTable(reads,k,w)
        stats=NumReadsStats(kmers)
        stats["k"] = k
        stats["windows"]=w
        results.append(stats)

kmer_expWindows= pd.DataFrame(results)
print(kmer_expWindows)
#kmer_expWindows.to_csv("kmer_exp_windows.csv")

In [ ]:
#kmer_expWindows.to_csv("kmer_exp_windows.csv")
kmer_expWindows=pd.read_csv("kmer_exp_windows.csv")

In [ ]:
plt.figure(figsize=(8,6))
for w in sorted(kmer_expWindows["windows"].unique()):
    sub = kmer_expWindows[kmer_expWindows["windows"] == w].sort_values("k")
    plt.errorbar(sub["k"],sub["Mean_reads_kmer"],yerr=[sub["Mean_reads_kmer"]-sub["ci_lower"],sub["ci_upper"]-sub["Mean_reads_kmer"]],fmt='o-',capsize=4,label=f"windows={w}")
plt.xlabel("Kmer Length (nucleotides)")
plt.ylabel("Mean Reads Per Kmer")
plt.title("Mean Reads Per Kmer Vs. Kmer Length")
plt.xticks(sorted(kmer_expWindows["k"].unique()))
plt.legend(title="Window shift")
plt.show()

In [ ]:
Kmer_tables={}
len_kmer=[6,7]
windows=[0,1,2]
for k in tqdm(len_kmer):
    Kmer_tables[k]={}
    for w in windows:
        Kmer_tables[k][w]= get_NKmers(reads, k, w)

Kmer_tables